# Classical Speaker ID Pipeline — MFCC → SVM

**Goal:** Extract 120-dimensional MFCC feature vectors (20 coefficients + Δ + ΔΔ, mean + std) for every audio clip, train an SVM classifier, evaluate it, and save the model.

**Before running:**  
Place WAV recordings in `data/recordings/<speaker_name>/*.wav`  
Or generate synthetic placeholder data first:  
```bash
python scripts/generate_placeholder_data.py
```

## 1. Imports and configuration

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Make project root importable from the notebook
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from features.mfcc import extract_mfcc

DATA_DIR   = ROOT / 'data' / 'recordings'
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
print('ROOT:', ROOT)
print('Data dir:', DATA_DIR)

## 2. Load dataset

Each subdirectory of `data/recordings/` is a speaker label. We load every `.wav` file and extract its 120-dim MFCC descriptor.

In [ ]:
speaker_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])

if not speaker_dirs:
    raise RuntimeError(
        f'No speaker directories found in {DATA_DIR}.\n'
        'Run: python scripts/generate_placeholder_data.py'
    )

X_list, y_list = [], []

for spk_dir in speaker_dirs:
    wavs = sorted(spk_dir.glob('*.wav'))
    print(f'  {spk_dir.name}: {len(wavs)} clips', end=' ... ', flush=True)
    for wav in wavs:
        try:
            feat = extract_mfcc(str(wav))   # (120,)
            X_list.append(feat)
            y_list.append(spk_dir.name)
        except Exception as e:
            print(f'\n    skip {wav.name}: {e}')
    print('ok')

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list)

print(f'\nDataset: {X.shape[0]} clips × {X.shape[1]} features')
print(f'Classes: {sorted(set(y))}')

## 3. Encode labels and split 80 / 10 / 10

In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(y)

# First hold out 10% test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y_enc, test_size=0.10, stratify=y_enc, random_state=RANDOM_STATE
)
# Then 10% val from the remaining
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.111, stratify=y_trainval, random_state=RANDOM_STATE
)

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')
print('Classes:', le.classes_)

## 4. SVM baseline

We use a Pipeline `StandardScaler → SVC(RBF kernel)` so feature scaling is baked in and won't leak between train/test.  
A small grid search finds the best `C` and `gamma`.

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)),
])

param_grid = {
    'svm__C':     [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto', 0.001, 0.01],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
gs = GridSearchCV(pipe, param_grid, cv=cv, n_jobs=-1, verbose=1)
gs.fit(X_train, y_train)

print('\nBest params:', gs.best_params_)
print(f'CV accuracy: {gs.best_score_:.3f}')

best_model = gs.best_estimator_

## 5. Evaluation on validation set

In [ ]:
y_val_pred = best_model.predict(X_val)
print('=== Validation set ===')
print(classification_report(y_val, y_val_pred, target_names=le.classes_))

## 6. Final evaluation on held-out test set

In [ ]:
y_test_pred = best_model.predict(X_test)
print('=== Test set ===')
print(classification_report(y_test, y_test_pred, target_names=le.classes_))

## 7. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(5, 4), facecolor='#111')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_facecolor('#111')
ax.tick_params(colors='#aaa')
ax.xaxis.label.set_color('#aaa')
ax.yaxis.label.set_color('#aaa')
ax.title.set_color('#fff')
ax.set_title('SVM — Test Confusion Matrix')
plt.tight_layout()
plt.savefig(MODELS_DIR / 'svm_confusion.png', dpi=120, facecolor='#111')
plt.show()
print('Saved to models/svm_confusion.png')

## 8. Save model and class list

`scripts/serve.py` reads `models/svm.joblib` and `models/svm_classes.json` at startup.

In [ ]:
joblib.dump(best_model, MODELS_DIR / 'svm.joblib')
(MODELS_DIR / 'svm_classes.json').write_text(json.dumps(list(le.classes_)))

print('Saved:')
print(f'  models/svm.joblib')
print(f'  models/svm_classes.json  →  {list(le.classes_)}')
print('\nRe-start the server to pick up the new model.')